# 🚀 ML Trading Strategy for S&P 500 - Complete Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/ml-sp500-trading-strategy/blob/main/notebooks/Google_Colab_Complete_Pipeline.ipynb)

This notebook runs the complete ML trading strategy pipeline in Google Colab.

## What This Does:
1. ✅ Clone the repository
2. ✅ Install all dependencies
3. ✅ Download 10 years of market data
4. ✅ Engineer 60+ features (GARCH, technical indicators)
5. ✅ Train LSTM neural network
6. ✅ Run backtest and generate results
7. ✅ Display visualizations

## Estimated Runtime: ~15-20 minutes

---

**⚠️ DISCLAIMER**: This is for educational purposes only. NOT financial advice!

## Step 1: Setup Environment

In [ ]:
# Check if running in Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("✓ Running in Google Colab")
else:
    print("⚠ Not running in Colab. This notebook is optimized for Colab.")

In [ ]:
%%bash
# Clone repository
if [ ! -d "ml-sp500-trading-strategy" ]; then
    echo "Cloning repository..."
    git clone https://github.com/yourusername/ml-sp500-trading-strategy.git
    echo "✓ Repository cloned"
else
    echo "✓ Repository already exists"
fi

In [ ]:
# Change to project directory
import os
os.chdir('ml-sp500-trading-strategy')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies... (this may take 2-3 minutes)")
!pip install -q -r requirements.txt
print("✓ Dependencies installed")

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display, HTML
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

## Step 2: Download and Prepare Data

In [ ]:
import config
from src.data_sourcing import DataDownloader

print("Downloading market data...")
print(f"Date range: {config.START_DATE} to {config.END_DATE}")
print("")

# Initialize downloader
downloader = DataDownloader(config.START_DATE, config.END_DATE, config.DATA_DIR)

# Download all data
data = downloader.download_all(
    sp500_ticker=config.SP500_TICKER,
    vix_ticker=config.VIX_TICKER,
    sector_tickers=config.SECTOR_TICKERS
)

print("\n✓ Data download completed!")

In [ ]:
# Merge S&P 500 and VIX
from src.data_sourcing import merge_market_data

if data['sp500'] is not None and data['vix'] is not None:
    merged_data = merge_market_data(data['sp500'], data['vix'])
    downloader.save_data(merged_data, "market_data.parquet")
    
    print(f"Market data shape: {merged_data.shape}")
    print(f"Date range: {merged_data.index.min()} to {merged_data.index.max()}")
    print(f"Total trading days: {len(merged_data)}")
    display(merged_data.head())

## Step 3: Feature Engineering

In [ ]:
from src.feature_engineering import FeatureEngineer

print("Engineering features...")
print("This includes:")
print("- Technical indicators (SMA, EMA, RSI, MACD, Bollinger Bands, ATR)")
print("- GARCH volatility forecasting")
print("- Temporal features")
print("- Lagged features")
print("")

# Load market data
market_data = downloader.load_data("market_data.parquet")

# Create features
fe = FeatureEngineer(market_data)
features_df = fe.create_all_features(config)

# Save features
downloader.save_data(features_df, "features.parquet")

print("\n✓ Feature engineering completed!")
print(f"Total features: {len(fe.get_feature_names())}")
print(f"Dataset shape: {features_df.shape}")
print(f"\nTarget distribution:")
print(features_df['target'].value_counts(normalize=True))

## Step 4: Train LSTM Model

In [ ]:
from src.model import LSTMModel
from src.utils import set_random_seed

# Set random seed for reproducibility
set_random_seed(config.RANDOM_SEED)

print("Preparing data for LSTM model...")

# Get feature columns
exclude_cols = ['open', 'high', 'low', 'close', 'volume', 'adj close', 'target']
feature_cols = [col for col in features_df.columns if col.lower() not in exclude_cols]

print(f"Using {len(feature_cols)} features")

# Initialize model
lstm_model = LSTMModel(config)

# Prepare data
data = lstm_model.prepare_data(features_df, feature_cols)

print("\n✓ Data prepared for training")

In [ ]:
print("Building and training LSTM model...")
print("This may take 5-10 minutes depending on hardware.")
print("GPU will significantly speed up training.")
print("")

# Build model
lstm_model.build_model(input_shape=(config.SEQUENCE_LENGTH, len(feature_cols)))

# Train model
history = lstm_model.train(data)

print("\n✓ Model training completed!")

In [ ]:
# Evaluate on test set
print("Evaluating model on test set...")

metrics = lstm_model.evaluate(data['X_test'], data['y_test'])

# Save model
lstm_model.save_model()

print("\n✓ Model saved successfully")

In [ ]:
# Plot training history
from src.plotting import StrategyVisualizer

visualizer = StrategyVisualizer(config)
visualizer.plot_training_history(
    history,
    save_path="results/training_history.png"
)

# Display plot
display(Image(filename="results/training_history.png"))

## Step 5: Run Backtest

In [ ]:
from src.portfolio_optimization import PortfolioOptimizer
from src.backtesting import Backtester

print("Running backtest...")

# Make predictions on test set
y_pred, y_proba = lstm_model.predict(data['X_test'])

# Calculate test start index
test_start_idx = int(len(features_df) * (config.TRAIN_RATIO + config.VALIDATION_RATIO)) + config.SEQUENCE_LENGTH
test_dates = features_df.index[test_start_idx:test_start_idx + len(data['y_test'])]

# Create predictions DataFrame
predictions_df = pd.DataFrame({
    'date': test_dates,
    'actual': data['y_test'],
    'prediction': y_pred,
    'probability': y_proba
})

print(f"Test period: {test_dates[0]} to {test_dates[-1]}")
print(f"Total test days: {len(test_dates)}")

In [ ]:
# Load sector data
sector_data = downloader.load_data("sectors_raw.parquet")
sector_data_aligned = sector_data.loc[test_dates[0]:test_dates[-1]]

# Initialize optimizer and backtester
portfolio_optimizer = PortfolioOptimizer(config)
backtester = Backtester(config)

# Run backtest
results_df = backtester.run_backtest(
    predictions_df,
    sector_data_aligned,
    portfolio_optimizer
)

print("\n✓ Backtest completed!")

In [ ]:
# Calculate metrics
sp500_data = downloader.load_data("market_data.parquet")
sp500_test = sp500_data.loc[test_dates[0]:test_dates[-1], 'close']
benchmark_returns = sp500_test.pct_change().dropna()

backtest_metrics = backtester.calculate_metrics(
    results_df=results_df,
    benchmark_returns=benchmark_returns
)

# Save results
from src.utils import save_results, save_metrics

save_results(results_df, config.RESULTS_CSV_PATH)
save_metrics(backtest_metrics, config.METRICS_JSON_PATH)

print("✓ Results saved")

## Step 6: Results and Visualizations

In [ ]:
# Display performance metrics
from src.utils import print_metrics_summary

print_metrics_summary(backtest_metrics)

In [ ]:
# Generate all visualizations
print("Generating visualizations...")

strategy_returns = pd.Series(
    results_df['daily_return'].values,
    index=results_df['date'].values
)

# 1. Cumulative returns
visualizer.plot_cumulative_returns(
    strategy_returns,
    benchmark_returns,
    save_path=config.CUMULATIVE_RETURNS_PLOT
)

# 2. Predictions vs actual
visualizer.plot_predictions(
    predictions_df['date'].values,
    predictions_df['actual'].values,
    predictions_df['prediction'].values,
    predictions_df['probability'].values,
    save_path=config.PREDICTIONS_PLOT
)

# 3. Portfolio allocation
visualizer.plot_portfolio_allocation(
    backtester.weights_history,
    save_path=config.ALLOCATION_PLOT
)

# 4. Confusion matrix
visualizer.plot_confusion_matrix(
    predictions_df['actual'].values,
    predictions_df['prediction'].values,
    save_path=config.CONFUSION_MATRIX_PLOT
)

# 5. Drawdown
visualizer.plot_drawdown(
    strategy_returns,
    save_path=config.DRAWDOWN_PLOT
)

print("✓ All visualizations generated")

In [ ]:
# Display: Cumulative Returns
print("=" * 60)
print("CUMULATIVE RETURNS: Strategy vs Buy-and-Hold")
print("=" * 60)
display(Image(filename=config.CUMULATIVE_RETURNS_PLOT))

In [ ]:
# Display: Predictions vs Actual
print("=" * 60)
print("MODEL PREDICTIONS vs ACTUAL DIRECTION")
print("=" * 60)
display(Image(filename=config.PREDICTIONS_PLOT))

In [ ]:
# Display: Portfolio Allocation
print("=" * 60)
print("PORTFOLIO ALLOCATION OVER TIME")
print("=" * 60)
display(Image(filename=config.ALLOCATION_PLOT))

In [ ]:
# Display: Confusion Matrix
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
display(Image(filename=config.CONFUSION_MATRIX_PLOT))

In [ ]:
# Display: Drawdown
print("=" * 60)
print("DRAWDOWN ANALYSIS")
print("=" * 60)
display(Image(filename=config.DRAWDOWN_PLOT))

## Step 7: Download Results (Optional)

In [ ]:
# If you want to download results to your computer
if IN_COLAB:
    from google.colab import files
    import zipfile
    
    # Create zip file with all results
    !zip -r results.zip results/
    
    print("Results packaged! Click below to download:")
    files.download('results.zip')
else:
    print("Results are saved in the 'results/' directory")

## Summary

✅ **Pipeline Completed Successfully!**

You have successfully:
1. Downloaded 10 years of market data
2. Engineered 60+ features with GARCH volatility
3. Trained a bidirectional LSTM neural network
4. Optimized portfolio allocation
5. Backtested the strategy with realistic costs
6. Generated comprehensive visualizations

### Key Takeaways:
- The ML strategy shows potential for superior risk-adjusted returns
- LSTM can capture patterns in financial time series
- Portfolio optimization reduces concentration risk
- Proper backtesting reveals true out-of-sample performance

### Next Steps:
1. Experiment with different hyperparameters in `config.py`
2. Try other optimization methods (Risk Parity, HRP)
3. Add more features (sentiment, macroeconomic data)
4. Test different ML models (GRU, Transformers)

---

**⚠️ REMEMBER**: This is for educational purposes only. Never use this with real money without extensive testing and professional advice!

**Star the repo on GitHub if you found this useful!** ⭐
